# LIME Debug Notebook

Testing LIME explanations with the Ollama Med42 model.

In [ ]:
import sys
import os
import json
import requests
sys.path.append('..')

import numpy as np
import pandas as pd
from lime.lime_text import LimeTextExplainer
import matplotlib.pyplot as plt
from utils.medical_processor import MedicalTermProcessor

In [5]:
# Initialize processors
medical_processor = MedicalTermProcessor()

# Model configuration
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "llama3-med42-8b"

def get_model_response(text: str) -> str:
    """Get response from Ollama model"""
    try:
        payload = {
            "model": MODEL_NAME,
            "messages": [
                {"role": "user", "content": text}
            ]
        }
        response = requests.post(
            OLLAMA_URL,
            json=payload,
            timeout=30
        )
        if response.status_code == 200:
            return response.json()["message"]["content"]
        return ""
    except Exception as e:
        print(f"[LIME] Model error: {e}")
        return ""

# Test model connection
test_response = get_model_response("Test connection")
print("Model connection test:", "Success" if test_response else "Failed")

Environment initialized


In [ ]:
# Initialize LIME explainer
explainer = LimeTextExplainer(
    class_names=['not_relevant', 'relevant'],
    split_expression='\s+',
    random_state=42
)

def predictor_fn(texts):
    """Prediction function for LIME"""
    predictions = []
    print(f"[LIME] Processing {len(texts)} samples...")
    
    for text in texts:
        try:
            # Get model response
            response = get_model_response(text)
            
            # Calculate medical term presence
            text_words = set(text.lower().split())
            resp_words = set(response.lower().split())
            medical_overlap = len((text_words | resp_words) & medical_processor.medical_terms)
            
            # Calculate relevance
            if (medical_overlap > 0):
                relevance = min(0.5 + (medical_overlap * 0.1), 0.9)
            else:
                relevance = 0.1
                
            predictions.append([1 - relevance, relevance])
            
        except Exception as e:
            print(f"[LIME] Prediction error: {e}")
            predictions.append([0.5, 0.5])
            
    return np.array(predictions)

# Test predictor
test_pred = predictor_fn(["Patient has severe headache"])
print("\nTest prediction:", test_pred)

Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not defined
Error processing text: name 'MODEL_NAME' is not 

In [ ]:
def explain_text(text: str, num_features: int = 10) -> tuple:
    """Generate and visualize LIME explanation"""
    print(f"\nAnalyzing text: {text}")
    
    # Get model response
    response = get_model_response(text)
    print(f"Model response: {response[:200]}...")
    
    # Generate explanation
    exp = explainer.explain_instance(
        text,
        predictor_fn,
        num_features=min(num_features, len(text.split())),
        num_samples=100,
        labels=(1,)
    )
    
    # Get feature importance
    feature_importance = exp.as_list(label=1)
    
    # Create visualization
    plt.figure(figsize=(12, 6))
    features, weights = zip(*feature_importance)
    colors = ['red' if w > 0 else 'blue' for w in weights]
    
    plt.barh(range(len(weights)), weights, color=colors)
    plt.yticks(range(len(weights)), features)
    plt.xlabel('Impact')
    plt.title('Feature Importance Analysis')
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return feature_importance, response

In [ ]:
# Test cases
test_cases = [
    "Patient presents with severe headache and fever",
    "Blood test shows elevated HbA1c levels",
    "Chest x-ray indicates bilateral infiltrates"
]

# Test first case
importance, model_response = explain_text(test_cases[0])

# Display results
print("\nFeature Importance Summary:")
df = pd.DataFrame(importance, columns=['Feature', 'Importance'])
print(df.sort_values('Importance', ascending=False))

In [ ]:
def test_stability(text: str, runs: int = 3):
    """Test explanation stability across multiple runs"""
    results = []
    
    for i in range(runs):
        print(f"\nRun {i+1}/{runs}:")
        importance, _ = explain_text(text)
        results.append(pd.DataFrame(importance, columns=['Feature', 'Importance']))
    
    # Compare results
    print("\nStability Analysis:")
    for i, df in enumerate(results):
        print(f"\nRun {i+1}:")
        print(df.sort_values('Importance', ascending=False))
        
    return results

# Test stability
stability_results = test_stability(test_cases[1])